# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [ ]:
##
import os
#from langchain.agents import initialize_agent, AgentType, Tool
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
import tiktoken
from IPython.display import Markdown, display
from langchain_classic import hub
from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
from langchain_community.tools.tavily_search import TavilySearchResults


load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4.1-nano")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:2500])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
#tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
#agent = initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

In [46]:
# Create the search tool instance

tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Search the web for the most recent information. Use for current events and 2025 updates."
    )
]

# Get the React prompt from hub
prompt = hub.pull("hwchase17/react")

# Create the agent
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
# Your prompt here

prompt_1 = """

You are a healthcare AI research analyst.

Your task:
Investigate the latest advancements in generative AI applied to healthcare in 2025.

You MUST:
- Use TavilySafeSearch to retrieve the most recent 2025 updates.
- Follow the ReAct format strictly.
- Provide the structured Markdown report ONLY in the Final Answer section.

When you reach the final answer, format it as:

# Latest Generative AI in Healthcare (2025)

## Major Breakthroughs
- Bullet points

## Clinical Applications
- Bullet points

## Research & Innovation
- Bullet points

## Key Sources
- Bullet points

Remember:
- Do not output Markdown until Final Answer.
- Follow Thought / Action / Action Input / Observation steps correctly.
"""

# Run it
#response_1 = agent.run(prompt_1)

response_1 = agent_executor.invoke({"input": prompt_1})

print(response_1["output"])




> Entering new AgentExecutor chain...
Question:  
You are a healthcare AI research analyst.

Your task:  
Investigate the latest advancements in generative AI applied to healthcare in 2025.

You MUST:  
- Use TavilySafeSearch to retrieve the most recent 2025 updates.  
- Follow the ReAct format strictly.

When you reach Final Answer, format it as:

# Latest Generative AI in Healthcare (2025)

## Major Breakthroughs
- Bullet points

## Clinical Applications
- Bullet points

## Research & Innovation
- Bullet points

## Key Sources
- Bullet points

Remember:
- Follow Thought / Action / Action Input / Observation steps correctly.

Thought: I need to find the most recent information on advancements in generative AI in healthcare for 2025. I will perform a web search using TavilySafeSearch with a query focused on "generative AI healthcare 2025 updates."

Action: TavilySafeSearch

Action Input: "latest advancements in generative AI in healthcare 2025"
[{'title': 'Generative AI in clinical (

In [ ]:
display(Markdown(response_1["output"]))

# Latest Generative AI in Healthcare (2025)

## Major Breakthroughs
- Development of advanced generative models such as diffusion models, Vision-Language Models (VLMs), and large language models (LLMs) that synthesize realistic, privacy-preserving clinical data.
- Integration of retrieval-augmented generation (RAG) frameworks to improve accuracy and transparency in clinical AI applications.
- Breakthroughs in diagnostic AI, notably models like PopEVE for pinpointing harmful genetic variants and CRISPR-GPT for streamlining gene-editing experiment design.
- Use of generative AI to enhance medical imaging quality, reduce radiation exposure, and generate cost-effective alternative images.
- AI-driven de novo biomedical design, including protein generation and molecular scaffolding, accelerating drug discovery and molecular research.

## Clinical Applications
- Automated summarization of extensive patient histories and clinical notes to support faster decision-making.
- Generation of synthetic medical images (e.g., chest X-rays) for training, research, and reducing the need for large datasets.
- Enhanced electronic health record (EHR) analysis, including narrative summarization and extraction of key health insights.
- Support for personalized treatment planning through AI-generated insights from genomic and clinical data.
- AI-assisted diagnosis of rare genetic diseases, significantly reducing diagnostic times and improving accuracy.

## Research & Innovation
- Rapid advancements in high-quality data synthesis for training robust AI models while maintaining patient privacy.
- Development of AI tools that assist in molecular and genetic research, including protein and gene editing design.
- Increasing adoption of AI-powered virtual assistants to streamline administrative tasks and improve clinician productivity.
- Exploration of multimodal models combining imaging, text, and molecular data for comprehensive clinical insights.
- Growing focus on transparency, accuracy, and ethical deployment of generative AI in healthcare settings.

## Key Sources
- PMC article on clinical applications of generative AI (2020–2025): https://pmc.ncbi.nlm.nih.gov/articles/PMC12620437/
- Philips healthcare technology trends report (2024): https://www.philips.com/a-w/about/news/archive/features/2024/10-healthcare-technology-trends-for-2025.html
- HealthTech Magazine overview of 2025 AI trends: https://healthtechmagazine.net/article/2025/01/overview-2025-ai-trends-healthcare
- Purdue University news on AI in healthcare systems (February 2025): https://www.purdue.edu/newsroom/purduetoday/2026/Q1/in-print-application-of-generative-ai-in-healthcare-systems
- Alation blog on AI breakthroughs in healthcare (2025): https://www.alation.com/blog/ai-healthcare-breakthroughs-2025-innovations/

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [ ]:
# Your prompt here
prompt_2 = """
Considering the following context: You're an analyst reporting to a C-level exec, who asks for your research. You provide consise information and deliver clean markdown summaries in a professional tone
To answer the question: Track 2025s top emerging AI startups and their innovations. Provide company names and product highlights.
Follow these guidelines:
- Use TavilySafeSearch to retrieve the most recent 2025 updates.
- Cite your sources
- Follow the ReAct format strictly.
- Provide the structured Markdown report ONLY in the Final Answer section.
"""

# Run it

response_2 = agent_executor.invoke({"input": prompt_2})





> Entering new AgentExecutor chain...
Action: TavilySafeSearch  
Action Input: "Top emerging AI startups 2025 innovations"  [{'title': 'Top AI Startups in 2025', 'url': 'https://www.startupblink.com/blog/top-ai-startups/', 'content': 'This article highlights the Top AI Startups of 2025, ranked using StartupBlink’s proprietary SB Score—a data-driven metric powered by Crunchbase and Semrush. The SB Score considers indicators such as website traffic, funding, and quarterly team growth. While some of these startups are developing core AI technologies, others are innovating by integrating powerful platforms like OpenAI into their products and services.\n\nJoin our Research Membership for extensive industry data, tailored insights, and custom datasets that answer your research questions.\n\nBecome a Research Member\n\n### Top AI Unicorns in 2025 [...] Our Research Members can request the full list of industry unicorns and further details on them.\n\n## Top AI Startups in 2025\n\nOn our glo

In [54]:
response_2

{'input': "\nConsidering the following context: You're an analyst reporting to a C-level exec, who asks for your research. You provide consise information and deliver clean markdown summaries in a professional tone\nTo answer the question: Track 2025s top emerging AI startups and their innovations. Provide company names and product highlights.\nFollow these guidelines:\n- Use TavilySafeSearch to retrieve the most recent 2025 updates.\n- Cite your sources\n- Follow the ReAct format strictly.\n- Provide the structured Markdown report ONLY in the Final Answer section.\n\n\n",
 'output': '# Top Emerging AI Startups of 2025 and Their Innovations\n\n## 1. Anysphere\n- **Product:** Cursor – AI coding tool\n- **Highlights:** Assists engineers in editing code using natural language prompts, competing with GitHub Copilot. Founded in 2022, it has signed contracts worth over $100 million, with rapid growth.\n- **Source:** [StartupBlink](https://www.startupblink.com/blog/top-ai-startups/)\n\n## 2. 

In [ ]:
#from IPython.display import Markdown, display


display(Markdown(response_2["output"]))


# Top Emerging AI Startups of 2025 and Their Innovations

## 1. Anysphere
- **Product:** Cursor – AI coding tool
- **Highlights:** Assists engineers in editing code using natural language prompts, competing with GitHub Copilot. Founded in 2022, it has signed contracts worth over $100 million, with rapid growth.
- **Source:** [StartupBlink](https://www.startupblink.com/blog/top-ai-startups/)

## 2. OpenEvidence
- **Product:** AI-powered medical information search platform
- **Highlights:** Focuses on healthcare, providing AI chatbots for medical professionals. Raised a $200 million Series C in 2025, valued at $6 billion.
- **Source:** [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

## 3. Lila Sciences
- **Product:** Science superintelligence platform
- **Highlights:** Aims to advance scientific research with AI. Secured $350 million in Series A funding.
- **Source:** [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

## 4. Reflection AI
- **Product:** AI for enterprise data analysis
- **Highlights:** Valued at $8 billion after a $2 billion Series B round; specializes in enterprise AI solutions.
- **Source:** [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

## 5. Chai Discovery
- **Product:** AI models for biotech and drug discovery
- **Highlights:** Raised $130 million in Series B, valuation at $1.2 billion.
- **Source:** [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

## 6. Falanna
- **Product:** Generative media platform
- **Highlights:** Raised $140 million in Series D, valuation over $4.5 billion.
- **Source:** [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

## 7. Unconventional AI
- **Product:** Foundational AI systems
- **Highlights:** Raised $475 million in seed funding; valued at nearly $4.5 billion, focusing on rethinking computer foundations.
- **Source:** [TechCrunch](https://techcrunch.com/2026/01/19/here-are-the-49-us-ai-startups-that-have-raised-100m-or-more-in-2025/)

## 8. Eva
- **Product:** Digital twin platform for AI model training
- **Highlights:** Uses semiconductor tech to drastically reduce training time and costs, cutting Llama 3.1 training from 80 days to under 2 days.
- **Source:** [MIT Sloan](https://mitsloan.mit.edu/ideas-made-to-matter/10-mit-ai-startups-to-watch-2025)

## 9. Gaia AI
- **Product:** Forestry and wildfire risk analysis
- **Highlights:** Uses modular AI to analyze complex motions, reducing scrap rates and improving environmental management.
- **Source:** [MIT Sloan](https://mitsloan.mit.edu/ideas-made-to-matter/10-mit-ai-startups-to-watch-2025)

## 10. Unbox AI
- **Product:** Enterprise AI data privacy solutions
- **Highlights:** Focuses on secure on-premises AI deployment to address data privacy concerns.
- **Source:** [MIT Sloan](https://mitsloan.mit.edu/ideas-made-to-matter/10-mit-ai-startups-to-watch-2025)

---

**Note:** This list highlights some of the most promising and well-funded startups in the AI space for 2025, showcasing innovations across coding, healthcare, scientific research, enterprise solutions, and environmental management.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [59]:
# Your prompt here
prompt_3 = """
User Request: Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
Step 1: Analyze the user request.
Step 2: Synthesize a response that integrates only relevant information
Step 3: Provide the response in a structured Markdown report ONLY in the Final Answer section.
Structure the response including:
- Summary of main points
- Specific details from context
- Any limitations or uncertainties
"""

# Run it
response_3 = agent_executor.invoke({"input": prompt_3})




> Entering new AgentExecutor chain...
Thought: To provide an accurate comparison of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro, I need the most recent information on their key features, pricing, and reviews. Since this information may have been updated recently, I will perform a web search to gather current details. I will use TavilySafeSearch to find the latest data on both products.

Action: TavilySafeSearch
Action Input: "OpenAI ChatGPT Team vs Anthropic Claude Pro features, pricing, reviews 2024"
[{'title': 'Claude AI vs ChatGPT: A Practical Comparison', 'url': 'https://www.appypieautomate.ai/blog/claude-vs-chatgpt', 'content': 'sum up, OpenAI currently provides a more feature-rich and widely-supported API environment, while Anthropic offers an API with unique strengths (context length and potentially more stable pricing) and a growing support system. Neither is “bad” at support – it’s more about the maturity and breadth of the ecosystem around OpenAI’s offerings that give 

In [60]:
display(Markdown(response_3["output"]))

# Comparison of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro (2024)

## Summary of Main Points
- **Key Features:** Both platforms offer advanced language models with web search, voice modes, and multi-modal capabilities. Claude Pro excels in handling very large context windows, while ChatGPT offers a broader ecosystem with more integrations.
- **Pricing:** Both have similar monthly fees (~$20-$30 per user), but differ in additional features and context limits. Claude Pro emphasizes large context windows (up to 500,000 tokens), whereas ChatGPT Pro offers up to 128,000 tokens.
- **Reviews & User Feedback:** Both are praised for ease of use and state-of-the-art performance. OpenAI’s ecosystem is more mature, with broader API support and enterprise options. Claude Pro is noted for its large context handling and cost-effectiveness for large documents.

## Specific Details

### 1. **Features**
- **OpenAI ChatGPT Team:**
  - Models: GPT-4, GPT-4 Turbo, with multimodal capabilities (images, text, voice).
  - Context Window: Up to 128,000 tokens.
  - Collaboration: Shared workspaces, custom GPTs, enterprise API options.
  - Additional: Integration with Microsoft products, extensive API support, and on-premise options for large clients.
  - Reviews: Widely supported, mature ecosystem, strong enterprise support.

- **Anthropic Claude Pro:**
  - Models: Claude Sonnet 4 (most advanced), Claude 3.5 Haiku, Claude Opus 4.
  - Context Window: Up to 500,000 tokens (significantly larger than GPT-4).
  - Collaboration: Team plans with shared projects, centralized billing, and admin controls.
  - Additional: Focus on safety, stability, and handling large documents; supports web search, voice, and multi-modal inputs.
  - Reviews: Praised for handling large contexts, cost-effectiveness, and safety features.

### 2. **Pricing**
| Platform | Tier | Approximate Cost | Key Features |
|------------|--------|------------------|--------------|
| **ChatGPT** | Plus | $20/month | GPT-4 access, multimodal, 128k tokens, API support |
| | Pro | $200/month | Most powerful models, unlimited usage, advanced features |
| **Claude** | Pro | $20/month (annual) | Claude 3.5 Sonnet, 200K tokens, Artifacts, API access |
| | Max | $100-200/month | Higher usage, early access, large context (up to 500K tokens) |
| | Team | $25/user/month | Collaboration, shared projects, admin controls |

*Note:* Both platforms offer free tiers with limited features.

### 3. **Reviews & User Feedback**
- **OpenAI ChatGPT:** 
  - Strengths: Ecosystem maturity, API support, enterprise solutions, multimodal features.
  - Limitations: Smaller context window compared to Claude, higher costs for large-scale use.
- **Anthropic Claude Pro:**
  - Strengths: Exceptionally large context window, safety focus, cost-effective for large documents.
  - Limitations: Slightly less mature ecosystem, fewer integrations than OpenAI.

## Limitations & Uncertainties
- Exact user reviews and satisfaction ratings are limited; most feedback is from industry reports and tech articles.
- Pricing details may vary based on enterprise agreements and regional differences.
- The rapid evolution of AI models means features and prices could change; always verify with official sources.

---

**In conclusion**, both ChatGPT Team and Claude Pro are powerful AI tools suited for different needs:
- Choose **ChatGPT Team** for a mature ecosystem, extensive integrations, and enterprise support.
- Opt for **Claude Pro** if large context handling, safety, and cost-efficiency for large documents are priorities.

Let me know if you need further details or specific use-case recommendations!

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
